# MI Monitoring — Patient-Level Nested Cross-Validation — Model Comparison (LR, RF, XGBoost, HGB)

## 1

In [ ]:
!pip -q install xgboost

## 2

In [ ]:
import os, re, time, hashlib, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, roc_curve,
    precision_recall_fscore_support, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

import sklearn
print("Python:", os.sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)

## 3

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_files = [fn for fn in uploaded.keys() if fn.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("CSV not found.")
DATA_PATH = csv_files[0]

df_raw = pd.read_csv(DATA_PATH)
print("rows:", df_raw.shape[0], "cols:", df_raw.shape[1])
print("columns:", list(df_raw.columns)[:20])

## 4

In [ ]:
def norm_col(c: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(c).lower())

def pick_col(df, candidates):
    cand_norm = {norm_col(c) for c in candidates}
    for c in df.columns:
        if norm_col(c) in cand_norm:
            return c
    return None

mrn_col = pick_col(df_raw, ["MRN", "mrn", "patient_mrn"])
if mrn_col is None:
    raise ValueError("MRN column not found.")

def hash_id(x, salt="MI_PROJECT_SALT_V1"):
    s = f"{salt}::{str(x)}"
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:12]

df = df_raw.copy()
df["patient_id"] = df[mrn_col].apply(hash_id)

direct_id_norm = {norm_col(x) for x in ["MRN","PatEngName","PatientName","Name","SAMPLE ID","Sample ID","sample_id","id"]}
drop_cols = [c for c in df.columns if norm_col(c) in direct_id_norm]
df.drop(columns=drop_cols, inplace=True, errors="ignore")

print("dropped:", drop_cols)
print("rows:", df.shape[0], "cols:", df.shape[1])

## 5

In [ ]:
age_col = pick_col(df, ["age", "Age"])
visit_col = pick_col(df, ["visit_type", "viist_type", "VisitType"])
troponin_col = pick_col(df, ["Troponin_IH", "Troponin-IH", "Troponin"])
ckmb_col = pick_col(df, ["CK_MB", "CK-MB", "CKMB"])
dx_text_col = pick_col(df, ["DIAGNOSIS", "Diagnosis", "diagnosis_text"])
dx_code_col = pick_col(df, ["code", "Diagnosis_code", "ICD", "ICD10", "icd_code"])
date_col = pick_col(df, ["Diagnosis_Date", "diagnosis_date", "visit_date", "date"])

rename_map = {}
if age_col: rename_map[age_col] = "age"
if visit_col: rename_map[visit_col] = "visit_type"
if troponin_col: rename_map[troponin_col] = "troponin"
if ckmb_col: rename_map[ckmb_col] = "ckmb"
if dx_text_col: rename_map[dx_text_col] = "diagnosis_text"
if dx_code_col: rename_map[dx_code_col] = "diagnosis_code"
if date_col: rename_map[date_col] = "diagnosis_date"

df.rename(columns=rename_map, inplace=True)
print(rename_map)

## 6

In [ ]:
def to_num(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower().replace(",", "")
    s = re.sub(r"[^0-9eE\.+\-]", "", s)
    try:
        return float(s)
    except:
        return np.nan

for col in ["age","troponin","ckmb"]:
    if col in df.columns:
        df[col] = df[col].apply(to_num)

def make_target(row):
    code = str(row.get("diagnosis_code","")).upper()
    text = str(row.get("diagnosis_text","")).lower()
    icd_flag = code.startswith("I21") or code.startswith("I22")
    text_flag = ("myocardial infarction" in text) or ("acute myocardial infarction" in text) or ("stemi" in text) or ("nstemi" in text)
    if "history of" in text or "old mi" in text or "rule out" in text:
        text_flag = False
    return 1 if (icd_flag or text_flag) else 0

df["target_MI"] = df.apply(make_target, axis=1).astype(int)

pos = int(df["target_MI"].sum())
n = len(df)
print("rows:", n, "positives:", pos, "rate:", round(pos/n, 4))

group_y = df.groupby("patient_id")["target_MI"].max()
print("patients:", int(group_y.shape[0]), "pos_patients:", int(group_y.sum()), "neg_patients:", int((group_y==0).sum()))

## 7

In [ ]:
if "diagnosis_date" in df.columns:
    df["diagnosis_date"] = pd.to_datetime(df["diagnosis_date"], errors="coerce")
    df = df.sort_values(["patient_id","diagnosis_date"])
else:
    df = df.sort_values(["patient_id"])

df["visit_index"] = df.groupby("patient_id").cumcount() + 1

for lab in ["troponin","ckmb"]:
    if lab not in df.columns:
        df[lab] = np.nan
    grp = df.groupby("patient_id")[lab]
    df[f"{lab}_prev"] = grp.shift(1)
    df[f"{lab}_delta"] = df[lab] - df[f"{lab}_prev"]
    df[f"{lab}_max_prior"] = grp.transform(lambda s: s.shift(1).cummax())

if "diagnosis_date" in df.columns:
    prev = df.groupby("patient_id")["diagnosis_date"].shift(1)
    df["days_since_last_visit"] = (df["diagnosis_date"] - prev).dt.days
else:
    df["days_since_last_visit"] = np.nan

print("rows:", df.shape[0], "patients:", df["patient_id"].nunique())

## 8

In [ ]:
feature_cols = [c for c in [
    "age","visit_type","visit_index","days_since_last_visit",
    "troponin","troponin_prev","troponin_delta","troponin_max_prior",
    "ckmb","ckmb_prev","ckmb_delta","ckmb_max_prior"
] if c in df.columns]

X = df[feature_cols].copy()
y = df["target_MI"].copy()
groups = df["patient_id"].copy()

cat_cols = [c for c in X.columns if X[c].dtype == "object"]
num_cols = [c for c in X.columns if c not in cat_cols]

print("X:", X.shape)
print("num_cols:", num_cols)
print("cat_cols:", cat_cols)

## 9

In [ ]:
numeric_tf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_tf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_tf, num_cols),
        ("cat", categorical_tf, cat_cols)
    ],
    remainder="drop"
)

## 10

In [ ]:
class StratifiedGroupKFoldApprox:
    def __init__(self, n_splits=5, shuffle=True, random_state=42):
        self.n_splits = int(n_splits)
        self.shuffle = bool(shuffle)
        self.random_state = int(random_state)

    def split(self, X, y, groups):
        y = np.asarray(y)
        groups = np.asarray(groups)

        uniq_groups, inv = np.unique(groups, return_inverse=True)
        group_y = np.zeros(len(uniq_groups), dtype=int)
        for gi in range(len(uniq_groups)):
            group_y[gi] = int(y[inv == gi].max())

        n1 = int(group_y.sum())
        n0 = int(len(group_y) - n1)
        k = min(self.n_splits, n0, n1)
        if k < 2:
            raise ValueError("Insufficient class groups for splitting.")

        skf = StratifiedKFold(n_splits=k, shuffle=self.shuffle, random_state=self.random_state)
        for train_g, test_g in skf.split(np.arange(len(uniq_groups)), group_y):
            train_mask = np.isin(inv, train_g)
            test_mask  = np.isin(inv, test_g)
            yield np.where(train_mask)[0], np.where(test_mask)[0]

OUTER_SPLITS = 5
INNER_SPLITS = 3
outer_cv = StratifiedGroupKFoldApprox(n_splits=OUTER_SPLITS, shuffle=True, random_state=SEED)
print("outer_splits:", OUTER_SPLITS, "inner_splits:", INNER_SPLITS)

## 11

In [ ]:
TARGET_RECALL = 0.98

def choose_threshold_for_recall(y_true, proba, target_recall=TARGET_RECALL):
    precision, recall, thr = precision_recall_curve(y_true, proba)
    thr = np.append(thr, 1.0)
    idxs = np.where(recall >= target_recall)[0]
    if len(idxs) == 0:
        f1 = (2*precision*recall) / np.clip(precision+recall, 1e-12, None)
        best = int(np.nanargmax(f1))
        return float(thr[best]), False
    best = idxs[np.argmax(precision[idxs])]
    return float(thr[best]), True

def metrics_at_threshold(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    ap = average_precision_score(y_true, proba)
    roc = roc_auc_score(y_true, proba) if len(np.unique(y_true)) > 1 else np.nan
    p,r,f1,_ = precision_recall_fscore_support(y_true, pred, average="binary", zero_division=0)
    cm = confusion_matrix(y_true, pred)
    return ap, roc, p, r, f1, cm

## 12

In [ ]:
models = {
    "LogReg": (
        LogisticRegression(max_iter=4000, class_weight="balanced", random_state=SEED),
        {"clf__C": np.logspace(-3, 2, 12)}
    ),
    "RandomForest": (
        RandomForestClassifier(
            n_estimators=600, random_state=SEED, n_jobs=-1,
            class_weight="balanced_subsample"
        ),
        {"clf__max_depth": [3, 5, 8, None],
         "clf__min_samples_leaf": [1, 5, 10, 30],
         "clf__max_features": ["sqrt", "log2", None]}
    ),
    "HGB": (
        HistGradientBoostingClassifier(random_state=SEED),
        {"clf__max_depth": [3, 4, 5, None],
         "clf__learning_rate": [0.02, 0.03, 0.05, 0.08, 0.1],
         "clf__max_leaf_nodes": [15, 31, 63, 127],
         "clf__min_samples_leaf": [10, 20, 40, 80],
         "clf__l2_regularization": [0.0, 0.1, 1.0]}
    ),
    "XGBoost": (
        XGBClassifier(
            random_state=SEED,
            n_estimators=900,
            tree_method="hist",
            eval_metric="logloss",
            learning_rate=0.05,
            max_depth=4,
            subsample=0.9,
            colsample_bytree=0.9
        ),
        {"clf__max_depth": [3,4,5,6],
         "clf__learning_rate": [0.02, 0.03, 0.05, 0.08],
         "clf__subsample": [0.7, 0.85, 1.0],
         "clf__colsample_bytree": [0.7, 0.85, 1.0],
         "clf__min_child_weight": [1, 3, 5]}
    )
}

SEARCH_ITERS = 12
print("models:", list(models.keys()))
print("search_iters:", SEARCH_ITERS)

## 13

In [ ]:
def fmt_time(sec):
    sec = float(sec)
    if sec < 60:
        return f"{sec:.1f}s"
    m = int(sec // 60)
    s = sec - 60*m
    return f"{m:d}m{s:0.0f}s"

def nested_cv_for_model(name, estimator, space, n_iter=SEARCH_ITERS):
    outer_rows = []
    t_model = time.time()

    for fold, (tr_idx, te_idx) in enumerate(outer_cv.split(X, y, groups=groups), 1):
        t_fold = time.time()

        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
        g_tr = groups.iloc[tr_idx]

        inner_cv = StratifiedGroupKFoldApprox(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED + 100 + fold)
        inner_splits = list(inner_cv.split(X_tr, y_tr, groups=g_tr))

        pipe = Pipeline(steps=[("prep", preprocess), ("clf", estimator)])

        search = RandomizedSearchCV(
            estimator=pipe,
            param_distributions=space,
            n_iter=n_iter,
            scoring="average_precision",
            n_jobs=-1,
            cv=inner_splits,
            refit=True,
            random_state=SEED,
            verbose=1,
            error_score="raise"
        )
        search.fit(X_tr, y_tr)

        best_est = search.best_estimator_

        proba_tr_oof = cross_val_predict(
            best_est, X_tr, y_tr,
            cv=inner_splits,
            method="predict_proba", n_jobs=-1
        )[:,1]

        thr, hit = choose_threshold_for_recall(y_tr.values, proba_tr_oof, TARGET_RECALL)

        proba_te = best_est.predict_proba(X_te)[:,1]
        ap, roc, p, r, f1, cm = metrics_at_threshold(y_te.values, proba_te, thr)

        outer_rows.append({
            "model": name,
            "fold": fold,
            "thr": float(thr),
            "hit_train_target": bool(hit),
            "test_PR_AUC": float(ap),
            "test_ROC_AUC": float(roc),
            "test_precision": float(p),
            "test_recall": float(r),
            "test_f1": float(f1),
            "tp": int(cm[1,1]), "fp": int(cm[0,1]), "fn": int(cm[1,0]), "tn": int(cm[0,0]),
            "best_params": search.best_params_
        })

        print(f"{name} fold {fold}  recall={r:.4f}  PR_AUC={ap:.4f}  thr={thr:.3f}  time={fmt_time(time.time()-t_fold)}")

    df_out = pd.DataFrame(outer_rows)
    print(f"{name} done  mean_recall={df_out['test_recall'].mean():.4f}  mean_PR_AUC={df_out['test_PR_AUC'].mean():.4f}  time={fmt_time(time.time()-t_model)}")
    return df_out

all_results = []
for name, (est, space) in models.items():
    df_res = nested_cv_for_model(name, est, space, n_iter=SEARCH_ITERS)
    all_results.append(df_res)

results = pd.concat(all_results, ignore_index=True)
results

## 14

In [ ]:
summary = results.groupby("model").agg(
    mean_PR_AUC=("test_PR_AUC","mean"),
    std_PR_AUC=("test_PR_AUC","std"),
    mean_ROC_AUC=("test_ROC_AUC","mean"),
    std_ROC_AUC=("test_ROC_AUC","std"),
    mean_precision=("test_precision","mean"),
    std_precision=("test_precision","std"),
    mean_recall=("test_recall","mean"),
    std_recall=("test_recall","std"),
    mean_f1=("test_f1","mean"),
    std_f1=("test_f1","std"),
).reset_index()

summary = summary.sort_values(["mean_PR_AUC","mean_recall"], ascending=False)
summary

## 15

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(summary["mean_recall"], summary["mean_PR_AUC"])
for _, row in summary.iterrows():
    plt.text(row["mean_recall"], row["mean_PR_AUC"], row["model"])
plt.xlabel("Mean Recall (Outer)")
plt.ylabel("Mean PR-AUC (Outer)")
plt.show()

## 16

In [ ]:
best_model_name = summary.iloc[0]["model"]
best_model_name

## 17

In [ ]:
best_rows = results[results["model"] == best_model_name].copy()

best_params_list = best_rows["best_params"].tolist()
s = [json.dumps(p, sort_keys=True) for p in best_params_list]
from collections import Counter
mode_params = Counter(s).most_common(1)[0][0]
final_params = json.loads(mode_params)

final_estimator = models[best_model_name][0]
final_pipe = Pipeline(steps=[("prep", preprocess), ("clf", final_estimator)]).set_params(**final_params)
final_pipe.fit(X, y)

outer_splits_list = list(outer_cv.split(X, y, groups=groups))
proba_oof = cross_val_predict(final_pipe, X, y, cv=outer_splits_list, method="predict_proba", n_jobs=-1)[:,1]

oper_thr, oper_hit = choose_threshold_for_recall(y.values, proba_oof, TARGET_RECALL)
ap, roc, p, r, f1, cm = metrics_at_threshold(y.values, proba_oof, oper_thr)

print("model:", best_model_name)
print("threshold:", round(oper_thr, 3), "hit_target:", oper_hit)
print("PR_AUC:", round(ap, 4), "ROC_AUC:", round(roc, 4), "Precision:", round(p, 4), "Recall:", round(r, 4), "F1:", round(f1, 4))
cm

## 18

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(cm)
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(["0","1"]); ax.set_yticklabels(["0","1"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for (i,j), v in np.ndenumerate(cm):
    ax.text(j, i, str(v), ha="center", va="center")
plt.show()

## 19

In [ ]:
prec, rec, _ = precision_recall_curve(y.values, proba_oof)
fpr, tpr, _ = roc_curve(y.values, proba_oof)

plt.figure(figsize=(6,5))
plt.plot(rec, prec)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.show()

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.show()

## 20

In [ ]:
import joblib
os.makedirs("artifacts", exist_ok=True)

MODEL_PATH = "artifacts/mi_model_patient_nestedcv.joblib"
joblib.dump({
    "model": final_pipe,
    "feature_cols": feature_cols,
    "operational_threshold": float(oper_thr),
    "schema": {"num_cols": num_cols, "cat_cols": cat_cols},
}, MODEL_PATH)

print(MODEL_PATH)

In [ ]:
import re, hashlib
import numpy as np
import pandas as pd
import joblib

MODEL_PATH = "artifacts/mi_model_patient_nestedcv.joblib"
DATA_PATH  = "/content/diagnosis_cleaned.csv"

bundle = joblib.load(MODEL_PATH)
pipeline = bundle["model"]
thr = bundle["operational_threshold"]
feature_cols = bundle["feature_cols"]

df_raw = pd.read_csv(DATA_PATH)

def norm_col(c: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(c).lower())

def pick_col(df, candidates):
    cand_norm = {norm_col(c) for c in candidates}
    for c in df.columns:
        if norm_col(c) in cand_norm:
            return c
    return None

mrn_col = pick_col(df_raw, ["MRN", "mrn", "patient_mrn"])
if mrn_col is None:
    raise ValueError("MRN column not found in dataset.")

def hash_id(x, salt="MI_PROJECT_SALT_V1"):
    s = f"{salt}::{str(x)}"
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:12]

df = df_raw.copy()
df["patient_id"] = df[mrn_col].apply(hash_id)

direct_id_norm = {norm_col(x) for x in ["MRN","PatEngName","PatientName","Name","SAMPLE ID","Sample ID","sample_id","id"]}
drop_cols = [c for c in df.columns if norm_col(c) in direct_id_norm]
df.drop(columns=drop_cols, inplace=True, errors="ignore")

age_col = pick_col(df, ["age", "Age"])
visit_col = pick_col(df, ["visit_type", "viist_type", "VisitType"])
troponin_col = pick_col(df, ["Troponin_IH", "Troponin-IH", "Troponin"])
ckmb_col = pick_col(df, ["CK_MB", "CK-MB", "CKMB"])
date_col = pick_col(df, ["Diagnosis_Date", "diagnosis_date", "visit_date", "date"])

rename_map = {}
if age_col: rename_map[age_col] = "age"
if visit_col: rename_map[visit_col] = "visit_type"
if troponin_col: rename_map[troponin_col] = "troponin"
if ckmb_col: rename_map[ckmb_col] = "ckmb"
if date_col: rename_map[date_col] = "diagnosis_date"

df.rename(columns=rename_map, inplace=True)

def to_num(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower().replace(",", "")
    s = re.sub(r"[^0-9eE\.+\-]", "", s)
    try:
        return float(s)
    except:
        return np.nan

for col in ["age","troponin","ckmb"]:
    if col in df.columns:
        df[col] = df[col].apply(to_num)

if "diagnosis_date" in df.columns:
    df["diagnosis_date"] = pd.to_datetime(df["diagnosis_date"], errors="coerce")
    df = df.sort_values(["patient_id","diagnosis_date"])
else:
    df = df.sort_values(["patient_id"])

df["visit_index"] = df.groupby("patient_id").cumcount() + 1

for lab in ["troponin","ckmb"]:
    if lab not in df.columns:
        df[lab] = np.nan
    grp = df.groupby("patient_id")[lab]
    df[f"{lab}_prev"] = grp.shift(1)
    df[f"{lab}_delta"] = df[lab] - df[f"{lab}_prev"]
    df[f"{lab}_max_prior"] = grp.transform(lambda s: s.shift(1).cummax())

if "diagnosis_date" in df.columns:
    prev = df.groupby("patient_id")["diagnosis_date"].shift(1)
    df["days_since_last_visit"] = (df["diagnosis_date"] - prev).dt.days
else:
    df["days_since_last_visit"] = np.nan

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required engineered features: {missing}")

X_new = df[feature_cols].copy()

proba = pipeline.predict_proba(X_new)[:, 1]
pred  = (proba >= thr).astype(int)

out = pd.DataFrame({
    "patient_id": df["patient_id"].values,
    "risk_proba": proba,
    "pred": pred
})

print(out.head(20))
print("threshold:", thr)
print("pred_counts:", out["pred"].value_counts().to_dict())


In [ ]:

import re
import hashlib
import numpy as np
import pandas as pd
import joblib

# Load the trained model and parameters
MODEL_PATH = "artifacts/mi_model_patient_nestedcv.joblib"
DATA_PATH = "/content/diagnosis_cleaned.csv"

bundle = joblib.load(MODEL_PATH)
pipeline = bundle["model"]
operational_threshold = bundle["operational_threshold"]
feature_cols = bundle["feature_cols"]

# Load the dataset
df_raw = pd.read_csv(DATA_PATH)

# Function to normalize column names
def norm_col(c: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(c).lower())

# Function to pick the correct column from candidates
def pick_col(df, candidates):
    cand_norm = {norm_col(c) for c in candidates}
    for c in df.columns:
        if norm_col(c) in cand_norm:
            return c
    return None

# Find the MRN column and hash it to create patient_id
mrn_col = pick_col(df_raw, ["MRN", "mrn", "patient_mrn"])
if mrn_col is None:
    raise ValueError("MRN column not found in dataset.")

def hash_id(x, salt="MI_PROJECT_SALT_V1"):
    s = f"{salt}::{str(x)}"
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:12]

# Create a copy of the dataframe and add the patient_id column
df = df_raw.copy()
df["patient_id"] = df[mrn_col].apply(hash_id)

# Drop direct identifier columns
direct_id_norm = {norm_col(x) for x in ["MRN","PatEngName","PatientName","Name","SAMPLE ID","Sample ID","sample_id","id"]}
drop_cols = [c for c in df.columns if norm_col(c) in direct_id_norm]
df.drop(columns=drop_cols, inplace=True, errors="ignore")

# Rename columns to standard names
age_col = pick_col(df, ["age", "Age"])
visit_col = pick_col(df, ["visit_type", "viist_type", "VisitType"])
troponin_col = pick_col(df, ["Troponin_IH", "Troponin-IH", "Troponin"])
ckmb_col = pick_col(df, ["CK_MB", "CK-MB", "CKMB"])
date_col = pick_col(df, ["Diagnosis_Date", "diagnosis_date", "visit_date", "date"])

rename_map = {}
if age_col: rename_map[age_col] = "age"
if visit_col: rename_map[visit_col] = "visit_type"
if troponin_col: rename_map[troponin_col] = "troponin"
if ckmb_col: rename_map[ckmb_col] = "ckmb"
if date_col: rename_map[date_col] = "diagnosis_date"

df.rename(columns=rename_map, inplace=True)

# Convert numeric columns to numbers
def to_num(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower().replace(",", "")
    s = re.sub(r"[^0-9eE\.+\-]", "", s)
    try:
        return float(s)
    except:
        return np.nan

for col in ["age","troponin","ckmb"]:
    if col in df.columns:
        df[col] = df[col].apply(to_num)

# Sort and create visit_index
if "diagnosis_date" in df.columns:
    df["diagnosis_date"] = pd.to_datetime(df["diagnosis_date"], errors="coerce")
    df = df.sort_values(["patient_id","diagnosis_date"])
else:
    df = df.sort_values(["patient_id"])

df["visit_index"] = df.groupby("patient_id").cumcount() + 1

# Feature engineering for lab results
for lab in ["troponin","ckmb"]:
    if lab not in df.columns:
        df[lab] = np.nan
    grp = df.groupby("patient_id")[lab]
    df[f"{lab}_prev"] = grp.shift(1)
    df[f"{lab}_delta"] = df[lab] - df[f"{lab}_prev"]
    df[f"{lab}_max_prior"] = grp.transform(lambda s: s.shift(1).cummax())

if "diagnosis_date" in df.columns:
    prev = df.groupby("patient_id")["diagnosis_date"].shift(1)
    df["days_since_last_visit"] = (df["diagnosis_date"] - prev).dt.days
else:
    df["days_since_last_visit"] = np.nan

# Ensure all required features are present
missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required engineered features: {missing}")

# Define the MRN for the patient you want to test
mrn_to_test = "10065258"  # Replace with the actual MRN of the patient

# Generate the patient_id from the MRN
patient_id_to_test = hash_id(mrn_to_test)

# Filter the DataFrame for this specific patient_id
patient_visits_df = df[df["patient_id"] == patient_id_to_test]

# Check if there are visits for this patient
if patient_visits_df.empty:
    raise ValueError(f"No visits found for patient_id {patient_id_to_test}")

# Select the features for this patient
X_patient = patient_visits_df[feature_cols].copy()

# Make predictions
proba_patient = pipeline.predict_proba(X_patient)[:, 1]
pred_patient  = (proba_patient >= operational_threshold).astype(int)

# Output the results for this patient
for idx, (prob, pred) in enumerate(zip(proba_patient, pred_patient)):
    result_word = "احتشاء" if pred == 1 else "عدم احتشاء"
    print(f"زيارة {idx + 1}: احتمال الاحتشاء = {prob:.4f}, النتيجة = {result_word}")

# Summarize the overall result for the patient
final_result = "احتشاء" if pred_patient.any() else "عدم احتشاء"
print(f"\nالنتيجة النهائية للمريض {patient_id_to_test}: {final_result}")
